# Lesson 03 Lab — Baseline Measurement: Parameters, FLOPs, Latency, and Throughput

**Puzzle:** Which baseline numbers are required before a pruning result can be interpreted?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

A pruning percentage without a baseline is not a comparison. Parameters and analytical FLOPs describe model structure; median and tail latency describe a workload on one stack; throughput and peak memory answer still different questions. A useful baseline freezes all of them before changing the model.


## 0. Predict before running

1. Predict how batch 1 and batch 64 change latency and examples per second.
2. Explain why lower FLOPs does not mathematically guarantee lower p95 latency.
3. List every environment field required to compare a later pruned run.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

The concrete system is a three-layer CUDA MLP, two batch sizes, a known dtype, a fixed random input, a parameter counter, an analytical linear-FLOP ledger, repeated CUDA-event samples, and peak allocated memory.

- Structural metrics are deterministic for a frozen graph.
- Latency and throughput depend on the workload and timing protocol.
- Tail statistics require repeated samples, not one synchronized call.


## 2. Derive the mechanism

For linear layers, parameters are `in_features × out_features` plus bias and leading multiply-add work is `2 × batch × in × out`. These values are deterministic properties of the chosen shape. Latency is a distribution affected by warm-up, synchronization, and batch; throughput is `batch / elapsed_time` and cannot be inferred from a single-request timing. Peak allocated memory must be reset and sampled over the same measurement window.

### Mechanism at a glance

```mermaid
flowchart LR
  M["frozen dense model"] --> S["shape + parameter + FLOP ledger"]
  W["frozen workload grid"] --> H["reproducible timing harness"]
  M --> H
  H --> L["latency distribution"]
  H --> T["throughput"]
  H --> P["peak memory"]
  S --> B["baseline report"]
  L --> B
  T --> B
  P --> B
```

### Walk it step by step

1. **Freeze the graph before measuring.** Record every layer shape, dtype, parameter count, and analytical operation count before changing the model.
2. **Define workload points.** Batch size, input shape, sequence length, and concurrency belong to the baseline identity rather than to a footnote.
3. **Measure a distribution.** Warm the stack, synchronize device work, retain repeated latency samples, and reset the memory window.
4. **Keep metric meanings separate.** Parameters and FLOPs describe structure; latency, throughput, and peak memory describe one execution path.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 3
LESSON_TITLE = 'Baseline Measurement: Parameters, FLOPs, Latency, and Throughput'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260811
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | the same dense MLP evaluated at batch 1 |
| Candidate | the same dense MLP evaluated at batch 64 |
| Held constant | model weights, hidden sizes, dtype, GPU, warm-up, repetitions, and input distribution |
| Measurements | parameters, analytical FLOPs, median/p95 latency, throughput, and peak allocated memory |
| Evidence | `pytorch-gpu` |

**Experiment:** Record a complete dense MLP baseline at batch 1 and batch 64 with structural and runtime metrics.


## 5. Read the experiment code

The notebook computes the structural ledger directly from module shapes, then uses the same timing helper for both batches. CUDA synchronization happens after the event pair, and the result retains every sample so p95 can be recomputed. The batch comparison is not a candidate victory; it demonstrates why service workload belongs in the baseline identity.

Do not execute until the code implements the frozen table above.


In [2]:
dtype = torch.bfloat16
model = nn.Sequential(
    nn.Linear(1024, 2048, device=DEVICE, dtype=dtype), nn.GELU(),
    nn.Linear(2048, 1024, device=DEVICE, dtype=dtype), nn.GELU(),
    nn.Linear(1024, 256, device=DEVICE, dtype=dtype),
).eval()

def linear_flops(module, batch):
    return int(sum(2 * batch * m.in_features * m.out_features for m in module.modules() if isinstance(m, nn.Linear)))

def run_batch(batch):
    inp = torch.randn(batch, 1024, device=DEVICE, dtype=dtype)
    torch.cuda.reset_peak_memory_stats()
    times = timing_summary(cuda_times(lambda: model(inp), warmup=8, repeats=40))
    peak = torch.cuda.max_memory_allocated() / 2**20
    return inp, times, peak

x1, t1, peak1 = run_batch(1)
x64, t64, peak64 = run_batch(64)
metrics = {
    "parameters": count_params(model),
    "batch1_flops": linear_flops(model, 1),
    "batch64_flops": linear_flops(model, 64),
    "batch1_median_ms": t1["median_ms"],
    "batch1_p95_ms": t1["p95_ms"],
    "batch1_examples_s": 1000.0 / t1["median_ms"],
    "batch64_median_ms": t64["median_ms"],
    "batch64_p95_ms": t64["p95_ms"],
    "batch64_examples_s": 64000.0 / t64["median_ms"],
    "peak_memory_mib": max(peak1, peak64),
    "samples": {"batch1": t1["samples_ms"], "batch64": t64["samples_ms"]},
}
analysis = (
    f"The frozen MLP contains {metrics['parameters']:,} parameters and {metrics['batch1_flops']:,} "
    f"leading linear FLOPs at batch 1. Batch 1 measured median/p95 "
    f"{metrics['batch1_median_ms']:.6f}/{metrics['batch1_p95_ms']:.6f} ms, while batch 64 measured "
    f"{metrics['batch64_median_ms']:.6f} ms and {metrics['batch64_examples_s']:.1f} examples/s. "
    "The batch field therefore changes the meaning of the performance baseline even though the parameters are identical."
)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Parameters | 4,459,776 |
| Batch-1 FLOPs | 8,912,896 |
| Batch-1 median | 0.056288 ms |
| Batch-1 p95 | 0.059221 ms |
| Batch-64 median | 0.056608 ms |
| Batch-64 throughput | 1,130,582.3/s |
| Peak memory | 41.133 MiB |


## 7. Interpret rather than merely print

The frozen MLP contains 4,459,776 parameters and 8,912,896 leading linear FLOPs at batch 1. Batch 1 measured median/p95 0.056288/0.059221 ms, while batch 64 measured 0.056608 ms and 1130582.3 examples/s. The batch field therefore changes the meaning of the performance baseline even though the parameters are identical.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 3,
    "title": 'Baseline Measurement: Parameters, FLOPs, Latency, and Throughput',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Parameters, FLOPs, latency, throughput, and memory are complementary baseline fields, not interchangeable compression scores.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 3,
  "title": "Baseline Measurement: Parameters, FLOPs, Latency, and Throughput",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260811
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "parameters": 4459776,
    "batch1_flops": 8912896,
    "batch64_flops": 570425344,
    "batch1_median_ms": 0.05628800019621849,
    "batch1_p95_ms": 0.059220799244940274,
    "batch1_examples_s": 17765.77594716505,
    "batch64_median_ms": 0.05660799890756607,
    "batch64_p95_ms": 0.060395201295614244,
    "batch64_examples_s": 1130582.2716769085,
    "peak_memory_mib": 41.13330078125,
    "samples": {
      "batch1": [
        0.0724480003118515,
        0.06143999844789505,
        0.05843200162053108,
        0.05910399928689003,
        0.05814399942755699,
        0.058240000158548355,
        0.05734400078654289,
        0.056703999638557434

## 9. Make the bounded decision

> Parameters, FLOPs, latency, throughput, and memory are complementary baseline fields, not interchangeable compression scores.

**Acceptance/rollback:** Reject any pruning comparison that cannot reproduce the dense baseline within a predefined tolerance on the same hardware and software stack.

**Failure analysis:** Timing before warm-up can include allocator and kernel initialization. Dividing batch by host wall time without synchronization can overstate throughput. Peak memory from an earlier operation can contaminate the window. A baseline report should make each of these failure modes auditable.


## 10. Extend the evidence

Add power, cold-start, and operator traces, then repeat across a batch/sequence grid. Use confidence intervals or repeated runs when the acceptance margin is close to noise.

The full evidence boundary and references are in [`README.md`](README.md).
